# 파지 강도 동적 계산 — v2

**담당**: 송다현, 정수진 (동적 강도 파지)

v1(`grasp.ipynb`)을 실기 검증 결과에 맞춰 재작성했습니다. 원본은 그대로 두었습니다.
근거는 [`REVIEW.md`](REVIEW.md) 참고.

## v1 과 무엇이 다른가

| | v1 | v2 |
|---|---|---|
| **무게** | "잴 수 없다" → 폭으로 추측 (`F = C·w²`) | **실측합니다** (`actual_motor_torque`) |
| **힘 단위** | 뉴턴(N)을 그대로 `current` 에 넣음 (버그) | **`current` 로 직접 회귀** — 뉴턴을 안 거침 |
| **force closure** | `is_force_closure()` — 수학적으로 항상 False | **삭제** (직육면체 상자에는 과설계) |
| **접촉점** | `box[0], box[2]` = 대각선 꼭짓점 (버그) | 마주보는 변의 **중점** |
| **흐름** | 집기 → 끝 | **약하게 집기 → 무게 재기 → 힘 계산 → 다시 조이기** |

## 왜 무게를 잴 수 있게 됐나

E0509 에는 **관절토크 센서도 F/T 센서도 없습니다.** 그래서
`get_tool_force()` / `get_workpiece_weight()` 는 **구조적으로 항상 0** 입니다.

유일한 실측값은 **`actual_motor_torque`** (모터 전류 → 토크)이고,
이건 `read_data_rt()` 로만 읽을 수 있습니다.

아이폰 13(174 g)을 노이즈의 10배 SNR 로 검출했습니다. **분해능 20~50 g.**

## 왜 뉴턴을 안 쓰나

물리식 `F ≥ k·M·g / (2·μ)` 는 **"필요한 힘이 무게에 비례한다"는 형태**만 알려줍니다.
그런데 그리퍼는 뉴턴이 아니라 `current` 를 받고, **N ↔ current 변환은 알 수 없습니다.**

그래서 중간에 뉴턴을 거치지 않고 이렇게 씁니다.

```
current = A · 질량(kg) + B
          ^^^^^^^^^^^^^^^^^
          물리가 형태를 주고, 실험이 A·B 를 준다
```

`B` 는 무게가 0 이어도 필요한 최소 파지력입니다.

## 1. 연결

로봇 드라이버와 그리퍼 서버가 떠 있어야 합니다.

```bash
./scripts/robot_up.sh
```

In [ ]:
import time
import json
import statistics
from pathlib import Path

import numpy as np
import rclpy

import DR_init

ROBOT_ID, ROBOT_MODEL = "dsr01", "e0509"
DR_init.__dsr__id = ROBOT_ID
DR_init.__dsr__model = ROBOT_MODEL

rclpy.init()
node = rclpy.create_node("grasp_v2", namespace=ROBOT_ID)
DR_init.__dsr__node = node          # ← DSR_ROBOT2 import 보다 먼저여야 함

from DSR_ROBOT2 import (
    posx, posj, movel, movej, wait, DR_BASE,
    set_robot_mode, get_robot_mode, get_current_posx,
    set_tool, set_tcp, get_tool, get_tcp,
    read_data_rt,
    ROBOT_MODE_MANUAL, ROBOT_MODE_AUTONOMOUS,
)
from dsr_gripper import gripper_open, gripper_close, gripper_cmd

print("연결 완료. 현재 모드:", get_robot_mode(), "(1=autonomous)")

## 2. 툴 등록 — 이걸 안 하면 무게 측정이 안 됩니다

컨트롤러가 팔 끝에 뭐가 달렸는지 모르면 중력 모델이 틀리고, 무게 측정이 불가능합니다.

**manual 모드에서만 등록됩니다.** autonomous 에서 시도하면 거부당합니다
(`this command can only be used in manual mode`).

> `cog`(무게중심)와 TCP 는 **아직 근사값**입니다. 그리퍼 도면으로 실측해서 채우세요.
> 무게 0.5 kg 은 ROBOTIS 공식 스펙이라 정확합니다.

In [ ]:
from dsr_msgs2.srv import ConfigCreateTool, ConfigCreateTcp

TOOL_NAME = "rh_p12_rn"
TCP_NAME = "rh_p12_rn_tcp"

TOOL_WEIGHT = 0.5              # kg — ROBOTIS RH-P12-RN 스펙 (확실)
TOOL_COG = [0.0, 0.0, 60.0]    # mm — TODO(실측): 플랜지 기준 무게중심
TCP_POS = [0.0, 0.0, 150.0, 0.0, 0.0, 0.0]   # TODO(실측): 손가락 끝까지


def _call(cli, req, timeout=10.0):
    fut = cli.call_async(req)
    rclpy.spin_until_future_complete(node, fut, timeout_sec=timeout)
    return fut.result()


def register_tool():
    '''그리퍼를 툴로 등록한다. manual 모드로 잠깐 바꿨다가 되돌린다.'''
    ns = f"/{ROBOT_ID}/dsr_controller2"
    tool_cli = node.create_client(ConfigCreateTool, f"{ns}/tool/config_create_tool")
    tcp_cli = node.create_client(ConfigCreateTcp, f"{ns}/tcp/config_create_tcp")
    for c in (tool_cli, tcp_cli):
        c.wait_for_service(timeout_sec=5.0)

    set_robot_mode(ROBOT_MODE_MANUAL)      # ← 등록은 manual 에서만
    time.sleep(2)

    r = ConfigCreateTool.Request()
    r.name, r.weight = TOOL_NAME, TOOL_WEIGHT
    r.cog = TOOL_COG
    r.inertia = [0.0] * 6
    ok_tool = _call(tool_cli, r)

    r = ConfigCreateTcp.Request()
    r.name, r.pos = TCP_NAME, TCP_POS
    ok_tcp = _call(tcp_cli, r)

    set_tool(TOOL_NAME)
    set_tcp(TCP_NAME)

    set_robot_mode(ROBOT_MODE_AUTONOMOUS)  # ← 그리퍼는 autonomous 에서만 동작
    time.sleep(2)

    print("tool =", get_tool(), " tcp =", get_tcp(), " mode =", get_robot_mode())
    if not get_tool():
        raise RuntimeError("툴 등록 실패 — manual 모드 전환이 안 됐을 수 있습니다")


register_tool()

## 3. 그리퍼 규약

| 값 | 의미 |
|---|---|
| `position` | `0` = 완전 닫힘 ~ `750` = 완전 열림 |
| `current` | **파지 힘.** 낮으면 놓치고, 높으면 상자를 찌그러뜨립니다 |

`current` 는 **뉴턴이 아니라 레지스터 raw 값**입니다. v1 의 가장 큰 버그가 이 둘을 섞은 것이었습니다.

In [ ]:
STROKE_CLOSED, STROKE_OPEN = 0, 750

CURRENT_MIN = 50        # 이보다 낮으면 아예 못 잡음
CURRENT_MAX = 500       # TODO(실험): 종이상자가 찌그러지기 시작하는 값. 보수적으로 시작
CURRENT_PROBE = 100     # 무게를 재려고 '살짝' 잡을 때의 힘


def clamp_current(c):
    return int(max(CURRENT_MIN, min(CURRENT_MAX, round(c))))


def grip(current):
    '''주어진 힘으로 닫는다.'''
    gripper_close(current=clamp_current(current))
    wait(1)


def release():
    gripper_open()
    wait(1)


print(f"파지 힘 범위: {CURRENT_MIN} ~ {CURRENT_MAX}")

## 4. 무게 측정 — 이 노트북의 핵심

**검증 완료** (2026-07-13, 아이폰 13 = 174 g).

E0509 에는 토크 센서가 없습니다. `raw_joint_torque` 는 사실 `gravity_torque` 와 동일한
**모델 계산값**이고, 그래서 `external_joint_torque = raw − gravity = 0` 이 항상 성립합니다.
`get_tool_force()` 가 영원히 0 인 이유입니다.

유일한 실측값은 **`actual_motor_torque`** (모터 전류 → 토크)입니다.

```
빈 그리퍼를 닫고 토크 측정      → 기준선
상자를 물고 같은 자세에서 측정  → 측정값
질량 = (측정값 − 기준선) × 계수
```

> ⚠️ **자세가 바뀌면 지렛대가 바뀝니다.** 기준선과 측정은 **반드시 같은 자세**에서.
> 픽 자세가 고정이라면 계수 하나로 충분합니다.

> ⚠️ **빈 그리퍼를 "닫은" 상태**가 기준선입니다. 여는 것과 닫는 것 사이에도
> 0.14 Nm 쯤 차이가 나므로(손가락이 움직이며 무게중심 이동), 조건을 맞춰야 합니다.

In [ ]:
TORQUE_SAMPLES = 40         # 샘플 수 (0.05초 간격 → 약 2초)
CALIB_PATH = Path("weight_calib.json")

# 실측 기준(2026-07-13, 아이폰 174g): ΔJ3 = -0.549 Nm → 약 317 g/Nm
# TODO(재캘리브레이션): 실제 픽 자세에서 calibrate_weight() 로 다시 구하세요.
WEIGHT_JOINT = 2            # J3 (0-based). 이 자세에서 가장 크게 반응한 관절
GRAMS_PER_NM = 317.0


def motor_torque(n=TORQUE_SAMPLES):
    '''actual_motor_torque 의 중앙값과 노이즈(표준편차)를 반환.'''
    rows = []
    for _ in range(n):
        d = read_data_rt()
        if d is not None:
            rows.append(list(d.actual_motor_torque))
        time.sleep(0.05)
    if not rows:
        raise RuntimeError("read_data_rt 가 데이터를 주지 않습니다")
    cols = list(zip(*rows))
    med = [statistics.median(c) for c in cols]
    sd = [statistics.pstdev(c) if len(c) > 1 else 0.0 for c in cols]
    return med, sd


class WeightSensor:
    '''모터 토크로 물체 무게를 잰다. 기준선은 '빈 그리퍼를 닫은' 상태에서 잡는다.'''

    def __init__(self):
        self.baseline = None
        self.noise = None
        self.grams_per_nm = GRAMS_PER_NM
        if CALIB_PATH.exists():
            self.grams_per_nm = json.loads(CALIB_PATH.read_text())["grams_per_nm"]
            print(f"저장된 캘리브레이션 로드: {self.grams_per_nm:.1f} g/Nm")

    def take_baseline(self):
        '''빈 그리퍼를 닫고 기준선을 잡는다. 픽 자세에서 실행하세요.'''
        grip(CURRENT_PROBE)
        time.sleep(1)
        self.baseline, self.noise = motor_torque()
        print(f"기준선 J{WEIGHT_JOINT+1} = {self.baseline[WEIGHT_JOINT]:+.3f} Nm "
              f"(노이즈 {self.noise[WEIGHT_JOINT]:.3f})")
        return self.baseline

    def measure(self):
        '''지금 물고 있는 물체의 질량(kg). 기준선과 같은 자세여야 한다.'''
        if self.baseline is None:
            raise RuntimeError("take_baseline() 을 먼저 실행하세요")
        med, sd = motor_torque()
        d = med[WEIGHT_JOINT] - self.baseline[WEIGHT_JOINT]
        noise = max(self.noise[WEIGHT_JOINT], sd[WEIGHT_JOINT])
        grams = abs(d) * self.grams_per_nm
        if abs(d) < 3 * noise:
            print(f"⚠ 변화({d:+.3f})가 노이즈({noise:.3f})의 3배 미만 — 신뢰할 수 없습니다")
        print(f"Δ토크 = {d:+.3f} Nm  →  {grams:.0f} g")
        return grams / 1000.0

    def calibrate(self, known_mass_kg):
        '''무게를 아는 물체를 물린 상태에서 실행 → g/Nm 계수를 구한다.'''
        med, _ = motor_torque()
        d = abs(med[WEIGHT_JOINT] - self.baseline[WEIGHT_JOINT])
        if d < 1e-3:
            raise RuntimeError("토크 변화가 없습니다. 물체가 정말 매달려 있나요?")
        self.grams_per_nm = (known_mass_kg * 1000) / d
        CALIB_PATH.write_text(json.dumps({"grams_per_nm": self.grams_per_nm}))
        print(f"캘리브레이션 완료: {self.grams_per_nm:.1f} g/Nm  (Δ={d:.3f} Nm)")
        return self.grams_per_nm


scale = WeightSensor()

## 5. 티칭 — 픽 자세 저장

**무게 측정은 자세에 의존합니다.** 픽 자세를 정하고, 기준선도 그 자세에서 잡아야 합니다.

In [ ]:
set_robot_mode(ROBOT_MODE_MANUAL)
wait(1)
print("직접교시로 팔을 Pick 위치로 옮긴 뒤 다음 셀을 실행하세요.")

In [ ]:
_x, _ = get_current_posx()
PICK_POS = [round(v, 3) for v in _x]
print("PICK_POS =", PICK_POS)

set_robot_mode(ROBOT_MODE_AUTONOMOUS)
wait(1)
print("autonomous 복귀. 모드:", get_robot_mode())

## 6. 무게 캘리브레이션

무게를 **아는** 물체로 `g/Nm` 계수를 구합니다. 픽 자세에서 하세요.

1. 그리퍼를 비우고 `scale.take_baseline()`
2. 아는 무게의 물체를 물리고 (예: 500 ml 물병 = 500 g)
3. `scale.calibrate(0.5)`

> 여러 무게로 반복해서 선형인지 확인하면 더 좋습니다.

In [ ]:
# 1) 빈 그리퍼 — 픽 자세에서
release()
scale.take_baseline()

In [ ]:
# 2) 무게를 아는 물체를 물린 뒤 실행
KNOWN_MASS_KG = 0.174       # 예: 아이폰 13 = 174 g

grip(CURRENT_PROBE)
time.sleep(1)
scale.calibrate(KNOWN_MASS_KG)

## 7. 파지 강도 계산

물리식이 **형태**를 줍니다.

```
미끄러지지 않으려면   2 · F · μ ≥ M · g
→                    F ≥ (k · M · g) / (2 · μ)      즉  F ∝ M
```

`current` 는 힘에 대략 비례하므로:

```
current = A · M(kg) + B
```

- `A`, `B` 는 **실험으로** 구합니다 (`fit()`).
- `B` 는 무게가 0 이어도 필요한 최소 파지력입니다.
- **뉴턴을 거치지 않습니다.** N ↔ current 변환을 모르기 때문입니다.

In [ ]:
SAFETY = 1.2        # 안전 계수. 최소값에 딱 맞추면 실제로는 미끄러집니다.

# TODO(실험): try_current() → accept → fit() 로 채우세요.
#             데이터가 없으면 fixed 모드(보수적 상수)로 동작합니다.
FIT_A = None        # current per kg
FIT_B = None        # 기본 파지력


class GraspPlanner:
    def __init__(self, fixed_current=250):
        self.fixed_current = fixed_current       # 데이터 없을 때 쓰는 보수적 값
        self.a, self.b = FIT_A, FIT_B

    def current_for(self, mass_kg=None, mode="auto"):
        '''파지에 쓸 current 를 정한다.

        mode="fixed"     항상 같은 값 (고정 강도 팀과 동일)
        mode="measured"  current = A·M + B  (실측 무게 기반) ← 진짜 2단계
        mode="auto"      계수가 있고 무게도 있으면 measured, 아니면 fixed
        '''
        if mode == "fixed" or self.a is None or mass_kg is None:
            if mode == "measured":
                print("⚠ 아직 회귀 계수가 없습니다 → fixed 로 대체")
            return clamp_current(self.fixed_current)

        c = (self.a * mass_kg + self.b) * SAFETY
        return clamp_current(c)

    def fit(self, trials):
        '''accept 된 (질량, 최소성공 current) 쌍으로 A, B 를 최소자승 회귀.'''
        ok = [t for t in trials if t.get("accepted")]
        if len(ok) < 2:
            print(f"accept 된 데이터가 {len(ok)}개 — 2개 이상 필요합니다")
            return None
        m = np.array([t["mass_kg"] for t in ok])
        c = np.array([t["current"] for t in ok])
        A = np.vstack([m, np.ones_like(m)]).T
        (self.a, self.b), *_ = np.linalg.lstsq(A, c, rcond=None)
        pred = self.a * m + self.b
        rmse = float(np.sqrt(np.mean((pred - c) ** 2)))
        print(f"current = {self.a:.0f}·M + {self.b:.0f}   "
              f"(데이터 {len(ok)}개, RMSE {rmse:.1f})")
        return self.a, self.b


planner = GraspPlanner()

## 8. 파지 시퀀스 — 피드백 루프

**v1 과 가장 크게 달라진 부분입니다.** 무게는 집어야 잴 수 있으므로, 순서가 바뀝니다.

```
① 약하게 집는다  (current = CURRENT_PROBE)
② 무게를 잰다                            ← 여기가 새로 생긴 단계
③ 무게에 맞는 파지력을 계산한다
④ 그 힘으로 다시 조인다                  ← 진짜 파지
⑤ 들어서 옮긴다
```

원래 파일명이었던 `physical_feedback_loop` 의 그 피드백 루프입니다.
v1 의 `GraspController.HOLD` 에 `# TODO` 로 비어 있던 자리입니다.

In [ ]:
LIFT_MM = 100.0
SPEED_L = 50


def move_to(pose, vel=SPEED_L):
    movel(posx(pose), vel=vel, acc=vel, ref=DR_BASE)
    wait(1)


def pick_with_weight_feedback(pick_pos, place_pos=None, mode="auto"):
    '''약하게 집고 → 무게를 재고 → 힘을 다시 정해서 조인 뒤 → 옮긴다.'''
    # ① 접근 + 약하게 집기
    release()
    move_to(pick_pos)
    grip(CURRENT_PROBE)
    time.sleep(1)

    # ② 무게 측정 (픽 자세 그대로 — 자세가 바뀌면 값이 틀어집니다)
    mass = scale.measure()

    # ③ 파지력 계산
    cur = planner.current_for(mass, mode=mode)
    print(f"질량 {mass*1000:.0f} g  →  파지 current {cur}")

    # ④ 그 힘으로 다시 조인다
    grip(cur)
    time.sleep(1)

    # ⑤ 들어올린다
    lifted = list(pick_pos)
    lifted[2] += LIFT_MM
    move_to(lifted)

    if place_pos is not None:
        move_to(place_pos)
        release()
        up = list(place_pos)
        up[2] += LIFT_MM
        move_to(up)

    return mass, cur

## 9. 실험 루프 — A, B 를 구하는 데이터를 만든다

**이 노트북에서 가장 먼저 해야 할 일입니다.** 계수가 없으면 `measured` 모드가 안 켜집니다.

1. `try_current(c)` — 그 힘으로 집어서 들어본다. 무게도 자동으로 기록된다.
2. 미끄러지는지 눈으로 본다.
3. `accept(i)` / `reject(i)` 로 판정한다. **버틴 것 중 가장 낮은 current** 를 accept 하세요.
4. 무게가 다른 물체로 반복한다. (최소 2종, 많을수록 좋음)
5. `planner.fit(trials)` → `A`, `B` 확정.

In [ ]:
TRIALS_PATH = Path("grasp_trials_v2.json")
trials = json.loads(TRIALS_PATH.read_text()) if TRIALS_PATH.exists() else []


def _save():
    TRIALS_PATH.write_text(json.dumps(trials, indent=2, ensure_ascii=False))


def try_current(current, pick_pos=None, note=""):
    '''주어진 current 로 집어 들어본다. 무게도 함께 기록.'''
    pick_pos = pick_pos or PICK_POS
    release()
    move_to(pick_pos)

    grip(CURRENT_PROBE)             # 먼저 약하게 집어서 무게를 잰다
    time.sleep(1)
    mass = scale.measure()

    grip(current)                   # 시험할 힘으로 다시 조인다
    time.sleep(1)

    lifted = list(pick_pos)         # 들어올린다
    lifted[2] += LIFT_MM
    move_to(lifted)

    t = {"index": len(trials), "mass_kg": mass, "current": int(current),
         "accepted": None, "note": note}
    trials.append(t)
    _save()

    print(f"\n[{t['index']}] 질량 {mass*1000:.0f} g, current {current}")
    print("  → 미끄러졌나요? accept(i) 또는 reject(i) 로 판정하세요.")
    return t["index"]


def review():
    print(f"{'idx':>3} | {'질량(g)':>8} | {'current':>7} | {'판정':>6} | note")
    for t in trials:
        v = {True: "버팀", False: "미끄러짐", None: "-"}[t["accepted"]]
        print(f"{t['index']:>3} | {t['mass_kg']*1000:>8.0f} | {t['current']:>7} | {v:>6} | {t['note']}")


def accept(i, note=""):
    trials[i]["accepted"] = True
    if note:
        trials[i]["note"] = note
    _save()
    print(f"✅ [{i}] 버팀 — 회귀에 사용됩니다")


def reject(i, note=""):
    trials[i]["accepted"] = False
    if note:
        trials[i]["note"] = note
    _save()
    print(f"❌ [{i}] 미끄러짐 — 기록만 남김")


print(f"이전 시도 {len(trials)}건 로드됨")

## 10. 실행

```python
# 실험 (계수 만들기)
try_current(150)      # 낮은 값부터
try_current(200)
try_current(250)
review()
accept(2)             # 버틴 것 중 가장 낮은 값
reject(0); reject(1)

# 무게가 다른 물체로 반복 → 최소 2종

planner.fit(trials)   # → A, B 확정

# 실제 파지 (무게 기반 동적 강도)
pick_with_weight_feedback(PICK_POS, mode="measured")
```

In [ ]:
# 예시 — 주석을 풀고 하나씩 실행하세요
#
# try_current(150)
# try_current(200)
# review()
# accept(1)
# reject(0)
# planner.fit(trials)
#
# pick_with_weight_feedback(PICK_POS, mode="measured")